In [2]:
import os
import glob
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Pre-Processing

### Feature Selection

In [1]:
FEATURE_COLUMNS = [
    "close",
    "open_price",
    "high",
    "low",
    "volume"
] 

In [9]:
df_bbca = pd.read_csv("Data/BBCA_5years.csv")
df_nvda = pd.read_csv("Data/NVDA_5years.csv")
df_tsla = pd.read_csv("Data/TSLA_5years.csv")

# Keep only the required columns
df_bbca = df_bbca[["date"] + FEATURE_COLUMNS].copy()
df_nvda = df_nvda[["date"] + FEATURE_COLUMNS].copy()
df_tsla = df_tsla[["date"] + FEATURE_COLUMNS].copy()

print("\nBBCA preview:")
print(df_bbca.head())

print("\nNVDA preview:")
print(df_nvda.head())

print("\nTSLA preview:")
print(df_tsla.head())


BBCA preview:
         date    close  open_price     high      low      volume
0  2019-07-29  30950.0     31100.0  31150.0  30800.0  13701900.0
1  2019-07-30  31100.0     31025.0  31250.0  30950.0   7645100.0
2  2019-07-31  30950.0     30950.0  31200.0  30900.0  12825800.0
3  2019-08-01  31050.0     30950.0  31350.0  30925.0  15645600.0
4  2019-08-02  30825.0     30800.0  31025.0  30800.0  12597900.0

NVDA preview:
         date    close  open_price     high      low     volume
0  2019-02-25  3.96725     4.06400  4.13200  3.95850  656020000
1  2019-02-26  3.92750     3.96250  4.01900  3.89375  488464000
2  2019-02-27  3.88525     3.91300  3.93975  3.82775  500256000
3  2019-02-28  3.85650     3.87500  3.89750  3.83400  324944000
4  2019-03-01  3.91125     3.90675  3.95375  3.84800  357188000

TSLA preview:
         date      close  open_price       high        low     volume
0  2019-02-25  19.917999   19.860666  20.193333  19.799999   99397500
1  2019-02-26  19.857332   19.481333  20.

### Min Max Scaler Function Init

In [5]:
class MinMaxScalerScratch:
    def __init__(self, eps=1e-8):
        self.min_ = None
        self.max_ = None
        self.eps = eps

    def fit(self, X):
        self.min_ = np.min(X, axis=0)
        self.max_ = np.max(X, axis=0)
        return self

    def transform(self, X):
        return (X - self.min_) / (self.max_ - self.min_ + self.eps)

    def fit_transform(self, X):
        return self.fit(X).transform(X)

    def inverse_transform_target(self, y_norm, target_col_idx=0):
        min_target = self.min_[target_col_idx]
        max_target = self.max_[target_col_idx]
        return y_norm * (max_target - min_target + self.eps) + min_target

### Sliding Window Function Init

In [6]:
def create_sliding_windows(data, window_size=5, target_col_idx=0):
    X, y = [], []
    num_rows, num_features = data.shape
    for i in range(window_size, num_rows):
        window = data[i - window_size:i, :]
        X.append(window.flatten())
        y.append(data[i, target_col_idx])
    return np.array(X), np.array(y).reshape(-1, 1)

### Scaler, Sliding Window, Data Split

In [10]:
import os
import glob
import numpy as np
import pandas as pd

# Define preprocessing parameters
TARGET_INDEX = 0
WINDOW_SIZE = 30

# Initialize dataset containers
X_train_list, y_train_list = [], []
X_val_list, y_val_list = [], []
X_test_list, y_test_list = [], []

val_records_by_ticker = {}
test_records_by_ticker = {}

# Define dataset directory
data_dir = r"Data"
stock_files = sorted(glob.glob(os.path.join(data_dir, "*.csv")))

# Define input features
FEATURE_COLUMNS = [
    "close",
    "open_price",
    "high",
    "low",
    "volume"
]

for file_path in stock_files:

    # Extract ticker name from filename
    ticker = os.path.basename(file_path).replace(".csv", "")

    # Load stock dataset
    df = pd.read_csv(file_path)

    # Convert date column to datetime format
    df["date"] = pd.to_datetime(df["date"])

    # Sort data chronologically
    df = df.sort_values("date").reset_index(drop=True)

    # Extract numerical features
    features = df[FEATURE_COLUMNS].to_numpy(dtype=float)

    # Skip datasets with insufficient observations
    if len(features) <= WINDOW_SIZE + 20:
        continue

    # Calculate chronological split indices
    n = len(features)
    train_end = int(n * 0.70)
    val_end = int(n * 0.85)

    # Separate training data
    train_raw = features[:train_end]

    # Initialize and fit scaler using training data only
    scaler = MinMaxScalerScratch()
    scaler.fit(train_raw)

    # Transform the entire dataset using training scaler
    all_scaled = scaler.transform(features)

    # Generate sliding windows from the complete dataset
    X_all, y_all = create_sliding_windows(
        all_scaled,
        window_size=WINDOW_SIZE,
        target_col_idx=TARGET_INDEX
    )

    # Split sliding windows based on their target positions
    X_tr = X_all[:train_end - WINDOW_SIZE]
    y_tr = y_all[:train_end - WINDOW_SIZE]

    X_val = X_all[train_end - WINDOW_SIZE:val_end - WINDOW_SIZE]
    y_val = y_all[train_end - WINDOW_SIZE:val_end - WINDOW_SIZE]

    X_te = X_all[val_end - WINDOW_SIZE:]
    y_te = y_all[val_end - WINDOW_SIZE:]

    # Extract validation and testing dates
    dates_val = df["date"].iloc[train_end:val_end].reset_index(drop=True)
    dates_test = df["date"].iloc[val_end:].reset_index(drop=True)

    # Store training datasets
    if len(X_tr) > 0:
        X_train_list.append(X_tr)
        y_train_list.append(y_tr)

    # Store validation datasets
    if len(X_val) > 0:
        X_val_list.append(X_val)
        y_val_list.append(y_val)

        val_records_by_ticker[ticker] = {
            "X_val": X_val,
            "y_val_norm": y_val,
            "scaler": scaler,
            "dates": dates_val,
            "actual_prices_raw": features[train_end:val_end, TARGET_INDEX]
        }

    # Store testing datasets
    if len(X_te) > 0:
        X_test_list.append(X_te)
        y_test_list.append(y_te)

        test_records_by_ticker[ticker] = {
            "X_test": X_te,
            "y_test_norm": y_te,
            "scaler": scaler,
            "dates": dates_test,
            "actual_prices_raw": features[val_end:, TARGET_INDEX]
        }

# Combine individual stock datasets into global datasets
X_train_global = np.vstack(X_train_list)
y_train_global = np.vstack(y_train_list)

X_val_global = np.vstack(X_val_list)
y_val_global = np.vstack(y_val_list)

X_test_global = np.vstack(X_test_list)
y_test_global = np.vstack(y_test_list)

# Display dataset preparation results
print("=== GLOBAL STOCK DATASET PREPARATION ===")

print(f"Number of Stocks Processed : {len(test_records_by_ticker)}")

print("\nTraining Dataset:")
print(f"X_train: {X_train_global.shape}")
print(f"y_train: {y_train_global.shape}")

print("\nValidation Dataset:")
print(f"X_val: {X_val_global.shape}")
print(f"y_val: {y_val_global.shape}")

print("\nTesting Dataset:")
print(f"X_test: {X_test_global.shape}")
print(f"y_test: {y_test_global.shape}")

=== GLOBAL STOCK DATASET PREPARATION ===
Number of Stocks Processed : 3

Training Dataset:
X_train: (2456, 150)
y_train: (2456, 1)

Validation Dataset:
X_val: (546, 150)
y_val: (546, 1)

Testing Dataset:
X_test: (546, 150)
y_test: (546, 1)
